In [0]:
# Service Principal credentials
application_id = "*****************"
authentication_key = "**********************"
tenant_id = "************************"

# Set Spark config for ADLS access
spark.conf.set("fs.azure.account.auth.type.delearn.dfs.core.windows.net", "OAuth")
spark.conf.set("fs.azure.account.oauth.provider.type.delearn.dfs.core.windows.net", "org.apache.hadoop.fs.azurebfs.oauth2.ClientCredsTokenProvider")
spark.conf.set("fs.azure.account.oauth2.client.id.delearn.dfs.core.windows.net", application_id)
spark.conf.set("fs.azure.account.oauth2.client.secret.delearn.dfs.core.windows.net", authentication_key)
spark.conf.set("fs.azure.account.oauth2.client.endpoint.delearn.dfs.core.windows.net", "https://login.microsoftonline.com/" + tenant_id + "/oauth2/token")

print("✅ Spark config set!")

✅ Spark config set!


In [0]:
dbutils.widgets.text("bronze_path", "")
dbutils.widgets.text("batch_id", "")
dbutils.widgets.text("source_system", "AzureSQL")
dbutils.widgets.text("silver_table_name", "scd_project.silver_customer_incremental")

bronze_path = dbutils.widgets.get("bronze_path")
batch_id = dbutils.widgets.get("batch_id")
source_system = dbutils.widgets.get("source_system")
silver_table_name = dbutils.widgets.get("silver_table_name")

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.window import Window

bronze_df = spark.read.parquet(bronze_path)

display(bronze_df)

In [0]:
silver_df = bronze_df \
    .withColumn("CustomerID", col("CustomerID").cast("int")) \
    .withColumn("CustomerName", trim(col("CustomerName"))) \
    .withColumn("City", trim(col("City"))) \
    .withColumn("Email", lower(trim(col("Email")))) \
    .withColumn("Phone", trim(col("Phone"))) \
    .withColumn("SourceSystem", lit(source_system)) \
    .withColumn("BatchID", lit(batch_id)) \
    .withColumn("IngestionDate", current_timestamp()) \
    .withColumn(
        "HashValue",
        sha2(
            concat_ws(
                "|",
                coalesce(col("CustomerName"), lit("")),
                coalesce(col("City"), lit("")),
                coalesce(col("Email"), lit("")),
                coalesce(col("Phone"), lit(""))
            ),
            256
        )
    )

In [0]:
window_spec = Window.partitionBy("CustomerID").orderBy(col("UpdatedDate").desc())

silver_final_df = silver_df \
    .withColumn("row_num", row_number().over(window_spec)) \
    .filter(col("row_num") == 1) \
    .drop("row_num") \
    .select(
        "CustomerID",
        "CustomerName",
        "City",
        "Email",
        "Phone",
        "CreatedDate",
        "UpdatedDate",
        "SourceSystem",
        "BatchID",
        "IngestionDate",
        "HashValue"
    )

In [0]:
spark.sql("CREATE DATABASE IF NOT EXISTS scd_project")

In [0]:
silver_path = f"abfss://sdcsilver@delearn.dfs.core.windows.net/customer/batch_id={batch_id}"

silver_final_df.write \
    .format("delta") \
    .mode("append") \
    .option("mergeSchema", "true") \
    .saveAsTable(silver_table_name)

silver_final_df.write \
    .mode("overwrite") \
    .parquet(silver_path)

---------------------------------------------------------------------------
NameError                                 Traceback (most recent call last)
File <command-8636202290616082>, line 3
      1 silver_path = "abfss://scdsilver@delearn.dfs.core.windows.net/silver/customer/"
----> 3 silver_final_df.write \
      4     .format("delta") \
      5     .mode("append") \
      6     .option("mergeSchema", "true") \
      7     .option("path", silver_path) \
      8     .saveAsTable(silver_table_name)

NameError: name 'silver_final_df' is not defined

In [0]:
display(
    spark.table(silver_table_name)
    .filter(col("BatchID") == batch_id)
    .orderBy("CustomerID")
)

print("Current batch silver count:", silver_final_df.count())